# 🔬 Xyether Anime Upscaler - Real-Time Model Benchmark & Visual Comparison
### Compare `Base net_g_89000`, `Phase 1 (2k & 8k)`, and `Xyether Strong v3 Final` on Google Colab GPU

In [ ]:
# @title 📦 1. Clone Engine & Setup Environment
import os, sys, subprocess

!rm -rf engine
!git clone https://github.com/Thanawanit/upscale-training.git engine
%cd engine

!pip install -q pytorch-msssim kornia huggingface_hub

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Environment Ready on {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})")


In [ ]:
# @title 📥 2. Download Checkpoints (Base, Step 2k, Step 8k, Final v3)
import urllib.request, os
from pathlib import Path

os.makedirs("/content/models", exist_ok=True)
HF_BASE = "https://huggingface.co/Thanawanit/Kaggle-Backup/resolve/main/checkpoints"

models_to_download = {
    "base_89k": f"{HF_BASE}/Phase4_XyetherKiller_V11/net_g_89000.pth",
    "p1_step2k": f"{HF_BASE}/Phase5_Xyether_Clone/xyether_phase1_step_2000.pth",
    "p1_step8k": f"{HF_BASE}/Phase5_Xyether_Clone/xyether_phase1_step_8000.pth",
    "final_v3": f"{HF_BASE}/Phase5_Xyether_Clone/Xyether_Strong_v3_Final.pth"
}

for name, url in models_to_download.items():
    dest = f"/content/models/{name}.pth"
    if not os.path.exists(dest):
        print(f"📡 Downloading {name}...")
        urllib.request.urlretrieve(url, dest)
        print(f"   Saved: {dest} ({os.path.getsize(dest)/(1024*1024):.2f} MB)")
    else:
        print(f"⚡ Already cached: {name}")

print("✅ All target models downloaded successfully!")


In [ ]:
# @title 🎨 3. Prepare Test Anime Image & Load Models
import cv2, requests
import numpy as np
import torch
from archs.xyether_compact import XyetherCompactNet, SRVGGNetCompact

def load_weights(model, path):
    ckpt = torch.load(path, map_location=device)
    sd = ckpt.get("params_ema", ckpt.get("params", ckpt))
    clean_sd = {k.replace("module.", ""): v for k, v in sd.items()}
    model.load_state_dict(clean_sd)
    model.eval()
    return model

# 1. Base Pretrained (True Base 89k with Nearest Residual)
model_base = XyetherCompactNet().to(device)
model_base = load_weights(model_base, "/content/models/base_89k.pth")

# 2. Phase 1 Step 2000 (Early Nearest Residual)
model_p1_2k = XyetherCompactNet().to(device)
model_p1_2k = load_weights(model_p1_2k, "/content/models/p1_step2k.pth")

# 3. Phase 1 Step 8000 (Fidelity Converged Nearest Residual)
model_p1_8k = XyetherCompactNet().to(device)
model_p1_8k = load_weights(model_p1_8k, "/content/models/p1_step8k.pth")

# 4. Final Xyether Strong v3 (Phase 2 GAN + FFLoss Polish)
model_final = XyetherCompactNet().to(device)
model_final = load_weights(model_final, "/content/models/final_v3.pth")

# Download high-res anime art test image
test_url = "https://safebooru.org/images/1104/56b92bb3f4dc78be9e9661671d438140.png"
resp = requests.get(test_url, timeout=15)
with open("/content/test_anime_hr.png", "wb") as f:
    f.write(resp.content)

hr_full = cv2.imread("/content/test_anime_hr.png")
h, w = hr_full.shape[:2]
cy, cx = h // 2, w // 2
crop_hr = hr_full[cy-256:cy+256, cx-256:cx+256]
cv2.imwrite("/content/crop_hr.png", crop_hr)

# Downscale 2x with area interpolation to create LR input
lr_input = cv2.resize(crop_hr, (256, 256), interpolation=cv2.INTER_AREA)
cv2.imwrite("/content/lr_input.png", lr_input)

print(f"✅ All 4 models loaded into {device}!")
print(f"✅ Test image prepared: Input LR (256x256), Target HR (512x512)")


In [ ]:
# @title 🚀 4. Run Quantitative Benchmark & Visual Comparison
from benchmark.xyether_evaluator import XyetherEvaluator
import matplotlib.pyplot as plt

evaluator = XyetherEvaluator(device=device)

models = {
    "Base 89k (Real)": model_base,
    "Phase 1 Step 2k": model_p1_2k,
    "Phase 1 Step 8k": model_p1_8k,
    "Strong v3 Final": model_final
}

print("=" * 65)
print(f"{'Model':<18} | {'Halo':<6} | {'Denoise':<8} | {'Line Depth':<10}")
print("=" * 65)

for name, m in models.items():
    rep = evaluator.evaluate_model(m)
    print(f"{name:<18} | {rep['overshoot_halo']:<6.2f} | {rep['noise_reduction_pct']:<7.1f}% | {rep['line_depth']:<10.2f}")
print("=" * 65)

# Run visual inference on LR image
lr_rgb = cv2.cvtColor(lr_input, cv2.COLOR_BGR2RGB)
lr_t = torch.from_numpy(lr_rgb.transpose(2, 0, 1)).float().unsqueeze(0).to(device) / 255.0

sr_dict = {}
with torch.no_grad():
    for name, m in models.items():
        out = m(lr_t).squeeze(0).permute(1, 2, 0).cpu().numpy()
        sr_dict[name] = (np.clip(out, 0, 1) * 255.0).astype(np.uint8)[:, :, ::-1]

nearest_bgr = cv2.resize(lr_input, (512, 512), interpolation=cv2.INTER_NEAREST)

# Comparison 1: Line Art Zoom Patch (128x128 around center)
p_near = nearest_bgr[192:320, 192:320]
p_base = sr_dict["Base 89k (Real)"][192:320, 192:320]
p_2k = sr_dict["Phase 1 Step 2k"][192:320, 192:320]
p_8k = sr_dict["Phase 1 Step 8k"][192:320, 192:320]
p_v3 = sr_dict["Strong v3 Final"][192:320, 192:320]
p_gt = crop_hr[192:320, 192:320]

line_grid = np.hstack([p_near, p_base, p_2k, p_8k, p_v3, p_gt])
cv2.imwrite("/content/comparison_line_grid.png", line_grid)

titles = ["Nearest 2x", "Base 89k (Real)", "Phase 1 (2k)", "Phase 1 (8k)", "Strong v3 Final", "Original 4K GT"]
patches = [p_near, p_base, p_2k, p_8k, p_v3, p_gt]

plt.figure(figsize=(22, 5))
for idx, (p, t) in enumerate(zip(patches, titles)):
    plt.subplot(1, 6, idx + 1)
    plt.imshow(cv2.cvtColor(p, cv2.COLOR_BGR2RGB))
    plt.title(t, fontsize=10)
    plt.axis("off")
plt.suptitle("🔬 Line Art Zoom Comparison", fontsize=14, weight="bold")
plt.tight_layout()
plt.show()

# Comparison 2: Flat / Color Area Zoom Patch (Checking for Checkerboard Dots)
p_near_c = nearest_bgr[50:178, 50:178]
p_base_c = sr_dict["Base 89k (Real)"][50:178, 50:178]
p_2k_c = sr_dict["Phase 1 Step 2k"][50:178, 50:178]
p_8k_c = sr_dict["Phase 1 Step 8k"][50:178, 50:178]
p_v3_c = sr_dict["Strong v3 Final"][50:178, 50:178]
p_gt_c = crop_hr[50:178, 50:178]

color_grid = np.hstack([p_near_c, p_base_c, p_2k_c, p_8k_c, p_v3_c, p_gt_c])
cv2.imwrite("/content/comparison_color_grid.png", color_grid)

color_patches = [p_near_c, p_base_c, p_2k_c, p_8k_c, p_v3_c, p_gt_c]
plt.figure(figsize=(22, 5))
for idx, (p, t) in enumerate(zip(color_patches, titles)):
    plt.subplot(1, 6, idx + 1)
    plt.imshow(cv2.cvtColor(p, cv2.COLOR_BGR2RGB))
    plt.title(t, fontsize=10)
    plt.axis("off")
plt.suptitle("🎨 Color & Shading Zoom Comparison (Checkerboard Check)", fontsize=14, weight="bold")
plt.tight_layout()
plt.show()

print("🎉 Line grid saved to: /content/comparison_line_grid.png")
print("🎉 Color grid saved to: /content/comparison_color_grid.png")
